# Zarr batch viz (single episode in MultiDataset)

This notebook builds a `MultiDataset` containing exactly one `ZarrDataset`, loads one batch, visualizes one image with `mediapy`, and prints the rest of the batch.

In [ ]:
from pathlib import Path
import torch
import mediapy as mpy

from egomimic.rldb.zarr.zarr_dataset_multi import ZarrDataset, MultiDataset
from egomimic.utils.egomimicUtils import EXTRINSICS
from egomimic.rldb.zarr.action_chunk_transforms import (
    _matrix_to_xyzypr,
    build_aria_bimanual_transform_list,
    build_eva_bimanual_transform_list,
)

import numpy as np
from egomimic.utils.egomimicUtils import INTRINSICS, draw_actions


In [ ]:
# Point this at a single episode directory, e.g. /path/to/episode_hash.zarr
EPISODE_PATH = Path("/coc/flash7/rco3/EgoVerse/egomimic/rldb/zarr/zarr/new/1769460905119.zarr")

key_map = {
    "images.front_1": {"zarr_key": "images.front_1"},
    "images.right_wrist": {"zarr_key": "images.right_wrist"},
    "images.left_wrist": {"zarr_key": "images.left_wrist"},
    "right.obs_ee_pose": {"zarr_key": "right.obs_ee_pose"},
    "right.obs_gripper": {"zarr_key": "right.gripper"},
    "left.obs_ee_pose": {"zarr_key": "left.obs_ee_pose"},
    "left.obs_gripper": {"zarr_key": "left.gripper"},
    "right.gripper": {"zarr_key": "right.gripper", "horizon": 45},
    "left.gripper": {"zarr_key": "left.gripper", "horizon": 45},
    "right.cmd_ee_pose": {"zarr_key": "right.cmd_ee_pose", "horizon": 45},
    "left.cmd_ee_pose": {"zarr_key": "left.cmd_ee_pose", "horizon": 45},
}

ACTION_CHUNK_LENGTH = 100
ACTION_STRIDE = 1  # set to 3 for Aria-style anchor sampling

extrinsics = EXTRINSICS["x5Dec13_2"]
left_extrinsics_pose = _matrix_to_xyzypr(extrinsics["left"][None, :])[0]
right_extrinsics_pose = _matrix_to_xyzypr(extrinsics["right"][None, :])[0]

transform_list = build_eva_bimanual_transform_list(
    chunk_length=ACTION_CHUNK_LENGTH,
    stride=ACTION_STRIDE,
    left_extra_batch_key={"left_extrinsics_pose": left_extrinsics_pose},
    right_extra_batch_key={"right_extrinsics_pose": right_extrinsics_pose},
)


In [ ]:
# Build a MultiDataset with exactly one ZarrDataset inside
single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map, transform_list=transform_list)
# single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map)
multi_ds = MultiDataset(datasets={"single_episode": single_ds}, mode="total")

print("len(single_ds):", len(single_ds))
print("len(multi_ds):", len(multi_ds))

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)


In [ ]:
def viz_batch(batch, image_key, action_key):
    # Image: (C,H,W) -> (H,W,C)
    img = batch[image_key][0].detach().cpu()
    if img.shape[0] in (1, 3):
        img = img.permute(1, 2, 0)
    img_np = img.numpy()

    # Make image drawable uint8
    if img_np.dtype != np.uint8:
        if img_np.max() <= 1.0:
            img_np = (img_np * 255.0).clip(0, 255).astype(np.uint8)
        else:
            img_np = img_np.clip(0, 255).astype(np.uint8)
    if img_np.shape[-1] == 1:
        img_np = np.repeat(img_np, 3, axis=-1)

    intrinsics = INTRINSICS["base"]
    actions = batch[action_key][0].detach().cpu().numpy()

    # 14D layout: [L xyz ypr g, R xyz ypr g]
    # 12D layout: [L xyz ypr, R xyz ypr]
    if actions.shape[-1] == 14:
        left_xyz = actions[:, :3]
        right_xyz = actions[:, 7:10]
    elif actions.shape[-1] == 12:
        left_xyz = actions[:, :3]
        right_xyz = actions[:, 6:9]
    else:
        raise ValueError(f"Unsupported action dim {actions.shape[-1]} for key {action_key}")

    vis = draw_actions(
        img_np.copy(), type="xyz", color="Blues",
        actions=left_xyz, extrinsics=None, intrinsics=intrinsics, arm="left"
    )
    vis = draw_actions(
        vis, type="xyz", color="Reds",
        actions=right_xyz, extrinsics=None, intrinsics=intrinsics, arm="right"
    )

    return vis



In [ ]:
image_key = "images.front_1"
action_key = "actions_cartesian"

for batch in loader:
    for k, v in batch.items():
        print(f"{k}: {tuple(v.shape)}")

    vis = viz_batch(batch, image_key=image_key, action_key=action_key)
    mpy.show_image(vis)
    break


In [ ]:
batch["actions_cartesian"][0, 0]


In [ ]:
batch["observations.state.ee_pose"][0]


## Aria Datasets

In [ ]:
from egomimic.utils.aws.aws_sql import timestamp_ms_to_episode_hash
timestamp_ms_to_episode_hash(1764285211791)

In [ ]:
# Aria-style chunking example: horizon=30 contiguous frames, sample anchors every 3 -> 10 points, then interpolate to 100.
EPISODE_PATH = Path("/coc/flash7/scratch/egoverseDebugDatasets/proc_zarr/1764285211791.zarr/")

key_map = {
    "images.front_1": {"zarr_key": "images.front_1"},
    "right.obs_ee_pose": {"zarr_key": "right.obs_ee_pose"},
    "left.obs_ee_pose": {"zarr_key": "left.obs_ee_pose"},
    "right.action_ee_pose": {"zarr_key": "right.obs_ee_pose", "horizon": 30},
    "left.action_ee_pose": {"zarr_key": "left.obs_ee_pose", "horizon": 30},
    "obs_head_pose": {"zarr_key": "obs_head_pose"},
}

ACTION_CHUNK_LENGTH = 100
ACTION_STRIDE = 3

transform_list = build_aria_bimanual_transform_list(
    chunk_length=ACTION_CHUNK_LENGTH,
    stride=ACTION_STRIDE,
    left_action_world="left.action_ee_pose",
    right_action_world="right.action_ee_pose",
)


In [ ]:
# Build a MultiDataset with exactly one ZarrDataset inside
# single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map, transform_list=transform_list)
single_ds = ZarrDataset(Episode_path=EPISODE_PATH, key_map=key_map)
multi_ds = MultiDataset(datasets={"single_episode": single_ds}, mode="total")

print("len(single_ds):", len(single_ds))
print("len(multi_ds):", len(multi_ds))

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)
# batch = next(iter(loader))

# print("Batch keys:", list(batch.keys()))


In [ ]:
batch = next(iter(loader))
print("Batch keys:", list(batch.keys()))
print(batch["right.action_ee_pose"][0, 0])
print(batch["left.action_ee_pose"][0, 0])
print(batch["obs_head_pose"][0])

In [ ]:
image_key = "images.front_1"
action_key = "actions_cartesian"

ims = []
for i, batch in enumerate(loader):
    first_img = batch[image_key][0].detach().cpu().permute(1, 2, 0).numpy()
    first_img = (first_img * 255.0).clip(0, 255).astype(np.uint8)

    vis = viz_batch(batch, image_key=image_key, action_key=action_key)
    ims.append(vis)
    # mpy.show_image(vis)

    # for k, v in batch.items():
    #     print(f"{k}: {tuple(v.shape)}")
    
    if i > 200:
        break

mpy.show_video(ims, fps=30)


In [ ]:
batch["actions_cartesian"][0, 0]


In [ ]:
batch[""]